### 제조,공업용 대시보드 제작 프로젝트
- 활용 데이터: AI4I 2020 Predictive Maintenance Dataset
- 제작 목적: 제작 공정 확인용 대시보드 제작
- 활용 라이브러리
    - Pandas
    - numpy
    - plotly
    - streamlit

- 제작일: 2026-10-15
- 버전: ver1

In [62]:
# 데이터 분석 및 가공
import pandas as pd
import numpy as np

# 데이터 시각화
import plotly as plt
import plotly.express as px
import seaborn as sns
import nbformat

# 대시보드 제작
import streamlit as st

### 1. 데이터 불러오기
- 데이터 불러오기
- 기본 정보 파악 
    - 결측치, 컬럼별 고유값 수 등

In [48]:
ai4i_rawdata= pd.read_csv("data/ai4i2020.csv")
ai4i_rawdata.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [49]:
ai4i_rawdata.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


### 2. 데이터 가공

In [50]:
### 파생변수 제작
#### 1. 섭씨온도 = 캘빈 온도  - 273.15
ai4i_rawdata['T1_C'] = ai4i_rawdata['Air temperature [K]'] - 273.15
#### 2. 온도차 = 공정온도 - 공기온도
ai4i_rawdata['T_distance'] = ai4i_rawdata['Process temperature [K]'] - ai4i_rawdata['Air temperature [K]']
#### 3. 동력(W) = 토크(Nm) * 회전수(rpm) * 2π / 60
ai4i_rawdata['Power'] = ai4i_rawdata['Torque [Nm]'] * ai4i_rawdata['Rotational speed [rpm]'] * 2 * np.pi / 60
#### 4. 마모 부하 지표 = 토크(Nm) / 회전수(rpm)
ai4i_rawdata['Wear load index'] = ai4i_rawdata['Torque [Nm]'] / ai4i_rawdata['Rotational speed [rpm]']  

In [54]:
### 데이터 중복 제거
ai4i_rawdata.drop_duplicates(inplace=True)
ai4i_cleaned = ai4i_rawdata.copy()

In [52]:
ai4i_rawdata[['Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 
       'Torque [Nm]', 'Tool wear [min]', 
       'T1_C', 'T_distance', 
       'Power', 'Wear load index']].describe()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],T1_C,T_distance,Power,Wear load index
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,300.004930,310.005560,1538.776100,39.986910,107.951000,26.854930,10.000630,6279.744953,0.026895
std,2.000259,1.483734,179.284096,9.968934,63.654147,2.000259,1.001094,1067.418295,0.008838
min,295.300000,305.700000,1168.000000,3.800000,0.000000,22.150000,7.600000,1148.440610,0.001317
25%,298.300000,308.800000,1423.000000,33.200000,53.000000,25.150000,9.300000,5561.184484,0.020640
50%,300.100000,310.100000,1503.000000,40.100000,108.000000,26.950000,9.800000,6271.027344,0.026735
75%,301.500000,311.100000,1612.000000,46.800000,162.000000,28.350000,11.000000,7003.002724,0.032805
max,304.500000,313.800000,2886.000000,76.600000,253.000000,31.350000,12.100000,10469.923005,0.063833


### 3. 통계 지표 확인
- 상관계수 분석
    - Tool wear [min] 와 토크, 회전속도(Wear load index)의 상관 관계 분석
        - 활용할 상관계수: 스피어만, 피어슨 
            - 스피어만(Spearman) : 비모수 통계, 단조관계 여부 측정
                - 값을 순위로 바꾸어 계산
            - 피어슨(Pearson) : 모수 통계, 직선 관계 측정
                - 실제 데이터 그대로 반영함으로 이상치에 매우 민감함

In [57]:
### 상관계수 계산
#### 1. 피어슨 상관계수
pearson_corr = ai4i_cleaned[[
                    "Tool wear [min]", "Wear load index"
                    ]].corr(method='pearson').iloc[0, 1]

spearman_corr = ai4i_cleaned[[
                    "Tool wear [min]", "Wear load index"
                    ]].corr(method='spearman').iloc[0, 1]

print(f"피어슨 상관계수: {pearson_corr:.3f}")
print(f"스피어만 상관계수: {spearman_corr:.3f}")

피어슨 상관계수: -0.003
스피어만 상관계수: -0.004


인사이트 : 2개 모두 약한 상관관계가 도출됨  
    - 따라서, 공구 마모 시간과 토크/회전속도에 대한 파생변수는 활용하지 않을 예정

In [74]:
# 4. 산점도 생성
ai4i_cleaned["Machine failure label"] = (
    ai4i_cleaned["Machine failure"]
    .map({0: "정상", 1: "고장"})
)

fig = px.scatter(
    ai4i_cleaned,
    x="Tool wear [min]",
    y="Wear load index",
    color="Machine failure label",
    color_discrete_map={"정상": "black", "고장": "red"},
    title="공구 마모 시간과 토크/회전속도의 관계",
    labels={
        "Tool wear [min]": "공구 마모 시간 (min)",
        "Wear load index": "토크 / 회전속도",
        "Machine failure label": "설비 상태",
    },
    opacity=0.65,
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="공구 마모 시간 (min)",
    yaxis_title="토크 / 회전속도 (Nm/rpm)",
)

fig.show(renderer="browser")

### 4. 대시보드 제작

In [75]:
#### 대시보드 제작용 데이터셋 저장
ai4i_cleaned.to_csv("data/ai4i_cleaned.csv", index=False)